# Part 5. ORM 관계 매핑

지금까지의 모델은 외래 키만 가지고 있었다. `Post.user_id`로 작성자의 ID에는 접근할 수 있지만, `post.author`로 작성자 객체에 직접 접근하지는 못했다. **관계(relationship)**가 그것을 가능하게 한다.

`relationship()`은 SQLAlchemy ORM의 가장 강력한 기능 중 하나다. 외래 키 위에 객체 그래프를 얹어, 자연스러운 Python 코드로 연관 데이터를 다룰 수 있게 한다.

In [ ]:
"""
이 예제에서 사용하는 테이블들과 관계

일대다 / 다대일 관계
	user - post   1:N
	user - profile 1:1  (1:N 의 특수한 관계)

다대다 관계
	user - group  N:M
	user - Membership - group  N:M (중간 테이블 적용시)

자기참조
	category (자기참조)
	User   (자기참조)  나를 팔로우 하는 사람들, 나를 팔로우 하는 사람들

상속매핑
	Employee
	└─ Engineer
	└─ Manager 

"""
None


# import

In [1]:
from typing import List
from sqlalchemy import String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

# engine

In [3]:
DB_USER = 'user2604'
DB_PASSWORD = '1234'
DB_HOST = 'localhost'
DB_PORT = 3306
DB_NAME = 'sqlalchemy_tutorial'

DATABASE_URL = (
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(DATABASE_URL, echo=True, pool_pre_ping=True)



# 🟡 도우미 함수        

In [4]:
from sqlalchemy.orm import Session
from sqlalchemy import select  

# User 테이블 조화
def print_user(session):    
    print('[Users]')
    for user in session.execute(select(User)).scalars():
        print(user)
    print()

def print_post(session):    
    print('[Posts]')
    for post in session.execute(select(Post)).scalars():
        print(post)
    print()

def print_group(session):    
    print('[Groups]')
    for group in session.execute(select(Group)).scalars():
        print(group)
    print()

def print_membership(session):    
    print('[Memberships]')
    for membership in session.execute(select(Membership)).scalars():
        print(membership)
    print()

def select_user_by_name(session, name):
    user = session.execute(select(User).where(User.name == name)).scalar_one_or_none()    
    return user

def select_post_by_title(session, title):
    post = session.scalars(select(Post).where(Post.title == title)).first()
    return post

def select_group_by_name(session, name):
    group = session.scalars(select(Group).where(Group.name == name)).first()
    return group

---

## 16장. 관계 기초

### 16.1 `relationship()`이란

`relationship()`은 두 매핑된 클래스를 연결하는 ORM 수준의 선언이다. 외래 키(`ForeignKey`)가 데이터베이스 수준의 관계라면, `relationship()`은 그 위에 있는 객체 수준의 관계다.

```python
from typing import List
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    posts: Mapped[List["Post"]] = relationship(back_populates="author")

class Post(Base):
    __tablename__ = "posts"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    title: Mapped[str] = mapped_column(String(200))
    
    author: Mapped["User"] = relationship(back_populates="posts")
```

이제 다음 같은 자연스러운 코드가 가능하다.

```python
user.posts                    # 사용자가 쓴 글 목록 (List[Post])
post.author                   # 글의 작성자 (User)
user.posts.append(new_post)   # 자동으로 user_id가 채워짐
post.author = some_user       # 자동으로 user_id가 갱신됨
```

### 16.2 `Mapped[T]`로 방향 추론

2.0 스타일에서는 `Mapped[T]`의 타입 힌트가 관계의 방향과 컬렉션 타입을 결정한다.

```python
posts: Mapped[List["Post"]] = relationship(...)  # 일대다 (컬렉션)  One To Many
author: Mapped["User"] = relationship(...)      # 다대일 (단일 객체)  Many To One
profile: Mapped["Profile | None"] = relationship(...)  # 선택적 일대일  Optional One To One
```

- `Mapped[List["X"]]` 또는 `Mapped[list["X"]]`: 일대다 (또는 다대다). `List["X"]` 안의 클래스 이름은 문자열이다. 클래스가 아직 정의되기 전에 참조되는 경우가 많기 때문이다.
- `Mapped["X"]`: 다대일 (또는 일대일). 단일 객체를 반환.
- `Mapped["X | None"]` 또는 `Mapped[Optional["X"]]`: NULL 허용. 관련 객체가 없을 수 있음.

`Set["X"]`나 `Dict[K, "X"]`도 사용할 수 있다. 컬렉션 클래스가 자동으로 적용된다.

### 16.3 `back_populates=`: 양방향 관계 연결하기

위 예제에서 양쪽 모두에 `back_populates`가 있다. 이는 두 `relationship()`이 **동일한 관계의 두 면**임을 SQLAlchemy에 알려주는 역할이다.

```python
# User 쪽
posts: Mapped[List["Post"]] = relationship(back_populates="author")
                                          # └── Post의 'author' 속성과 연결

# Post 쪽
author: Mapped["User"] = relationship(back_populates="posts")
                                     # └── User의 'posts' 속성과 연결
```

`back_populates`가 있으면 한쪽을 수정해도 다른 쪽이 자동으로 동기화된다.

```python
post = Post(title="Hello")
user.posts.append(post)
print(post.author)       # user (자동 설정됨)
print(post.user_id)      # 아직 None. flush 시 채워짐

# 반대 방향도 동작
new_post = Post(title="World")
new_post.author = user
print(user.posts)        # [..., new_post]가 포함됨
```

`back_populates` 없이 양쪽에 각각 `relationship()`을 선언하면 SQLAlchemy는 둘이 같은 관계인지 알지 못해, 한쪽만 갱신해도 다른 쪽은 동기화되지 않는다. 양방향 관계라면 **반드시 `back_populates`를 사용**한다.

### 16.4 `backref` (레거시)

1.x 시절에는 한쪽에서만 선언하고 `backref`로 반대편을 자동 생성하는 방식이 많이 쓰였다.

```python
# 1.x 스타일 (Legacy)
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    posts = relationship("Post", backref="author")  # Post.author가 자동 생성됨
```

`backref`는 2.0에서도 동작하지만, **타입 힌트와 호환되지 않는다**. Post 클래스에 `author` 속성이 명시적으로 정의되어 있지 않으므로 IDE와 타입 체커가 인식하지 못한다. 새 코드에서는 항상 `back_populates`를 사용하자.

### 16.5 외래 키와 관계의 위치

다대일 관계에서 외래 키는 항상 **"다(many)" 쪽**에 있다. `Post`가 여러 개 있을 수 있고 각 Post는 한 명의 User를 가리키므로, `user_id` 외래 키는 `Post` 테이블에 있다.

```python
class Post(Base):
    __tablename__ = "posts"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))  # FK는 여기
    title: Mapped[str]
    
    author: Mapped["User"] = relationship(back_populates="posts")
```

SQLAlchemy는 `ForeignKey("users.id")`를 보고 자동으로 조인 조건을 추론한다. 한 쌍의 테이블 사이에 외래 키가 하나뿐이라면 별도 설정 없이 동작한다. 외래 키가 여러 개거나 모호한 경우는 `primaryjoin`을 명시해야 한다(이는 고급 주제).

### 16.6 🟡종합 예제

#### User, Post

In [5]:
from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    posts: Mapped[List["Post"]] = relationship(back_populates="author")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r})"

class Post(Base):
    __tablename__ = "posts"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    title: Mapped[str] = mapped_column(String(200))
    
    author: Mapped["User"] = relationship(back_populates="posts")

    def __repr__(self) -> str:
        return f"Post(id={self.id!r}, title={self.title!r})"    

In [6]:
# 실습 혹시 이전에 테이블이 있었다면 삭제하고..
Base.metadata.drop_all(engine)

2026-06-01 17:20:47,920 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-06-01 17:20:47,921 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:20:47,922 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-06-01 17:20:47,923 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:20:47,924 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-06-01 17:20:47,924 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:20:47,925 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:20:47,925 INFO sqlalchemy.engine.Engine DESCRIBE `sqlalchemy_tutorial`.`users`
2026-06-01 17:20:47,926 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:20:47,929 INFO sqlalchemy.engine.Engine DESCRIBE `sqlalchemy_tutorial`.`posts`
2026-06-01 17:20:47,930 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:20:47,931 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(engine)

# CREATE TABLE users (
# 	id INTEGER NOT NULL AUTO_INCREMENT, 
# 	name VARCHAR(50) NOT NULL, 
# 	PRIMARY KEY (id)
# )

# CREATE TABLE posts (
# 	id INTEGER NOT NULL AUTO_INCREMENT, 
# 	user_id INTEGER NOT NULL, 
# 	title VARCHAR(200) NOT NULL, 
# 	PRIMARY KEY (id), 
# 	FOREIGN KEY(user_id) REFERENCES users (id)
# )

2026-06-01 17:21:06,413 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:21:06,414 INFO sqlalchemy.engine.Engine DESCRIBE `sqlalchemy_tutorial`.`users`
2026-06-01 17:21:06,414 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:21:06,415 INFO sqlalchemy.engine.Engine DESCRIBE `sqlalchemy_tutorial`.`posts`
2026-06-01 17:21:06,416 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-06-01 17:21:06,417 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	name VARCHAR(50) NOT NULL, 
	PRIMARY KEY (id)
)


2026-06-01 17:21:06,417 INFO sqlalchemy.engine.Engine [no key 0.00045s] {}
2026-06-01 17:21:06,447 INFO sqlalchemy.engine.Engine 
CREATE TABLE posts (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	user_id INTEGER NOT NULL, 
	title VARCHAR(200) NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(user_id) REFERENCES users (id)
)


2026-06-01 17:21:06,447 INFO sqlalchemy.engine.Engine [no key 0.00060s] {}
2026-06-01 17:21:06,497 INFO sqlalchemy.engine.Engine

#### User 한개 생성

In [8]:
with SessionLocal() as session:
    user = User(name = "Chris")
    
    session.add(user)
    session.commit()

    print_user(session)

2026-06-01 17:22:01,538 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:22:01,540 INFO sqlalchemy.engine.Engine INSERT INTO users (name) VALUES (%(name)s)
2026-06-01 17:22:01,540 INFO sqlalchemy.engine.Engine [generated in 0.00046s] {'name': 'Chris'}
2026-06-01 17:22:01,543 INFO sqlalchemy.engine.Engine COMMIT
[Users]
2026-06-01 17:22:01,550 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:22:01,552 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users
2026-06-01 17:22:01,552 INFO sqlalchemy.engine.Engine [generated in 0.00045s] {}
User(id=1, name='Chris')

2026-06-01 17:22:01,553 INFO sqlalchemy.engine.Engine ROLLBACK


#### User 가 Post 한개 작성 추가

In [9]:
# Chris 가 작성한 post 한개 추가
with SessionLocal() as session:
    user = session.execute(select(User).where(User.name == 'Chris')).scalar_one_or_none()
    post = Post(title='Python 뽀개기')
    post.user_id = user.id  # FK 설정
    session.add(post)
    session.commit()
    

2026-06-01 17:24:58,961 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:24:58,963 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users 
WHERE users.name = %(name_1)s
2026-06-01 17:24:58,964 INFO sqlalchemy.engine.Engine [generated in 0.00081s] {'name_1': 'Chris'}
2026-06-01 17:24:58,965 INFO sqlalchemy.engine.Engine INSERT INTO posts (user_id, title) VALUES (%(user_id)s, %(title)s)
2026-06-01 17:24:58,966 INFO sqlalchemy.engine.Engine [generated in 0.00072s] {'user_id': 1, 'title': 'Python 뽀개기'}
2026-06-01 17:24:58,968 INFO sqlalchemy.engine.Engine COMMIT


#### 특정 Post 의 User: Many -> One

In [13]:
# 특정 post 의 user   Many -> One
with SessionLocal() as session:
    post = select_post_by_title(session, "Python 뽀개기")
    # print(post)

    # 방법1 : FK 직접 사용
    print(post.user_id)
    user = session.get(User, post.user_id)
    print('🟠', user)

    # 방법2 : relation()
    print('🟣', post.author)

    

2026-06-01 17:29:37,214 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:29:37,215 INFO sqlalchemy.engine.Engine SELECT posts.id, posts.user_id, posts.title 
FROM posts 
WHERE posts.title = %(title_1)s
2026-06-01 17:29:37,215 INFO sqlalchemy.engine.Engine [cached since 179.2s ago] {'title_1': 'Python 뽀개기'}
1
2026-06-01 17:29:37,217 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name 
FROM users 
WHERE users.id = %(pk_1)s
2026-06-01 17:29:37,217 INFO sqlalchemy.engine.Engine [cached since 96.67s ago] {'pk_1': 1}
🟠 User(id=1, name='Chris')
🟣 User(id=1, name='Chris')
2026-06-01 17:29:37,218 INFO sqlalchemy.engine.Engine ROLLBACK


#### 특정 User 의 post : One -> Many

In [15]:
# 특정 User 의 post  
with SessionLocal() as session:
    user = select_user_by_name(session, "Chris")

    print(user.posts)  # posts 조회할때 추가 쿼리 발생

    

2026-06-01 17:32:10,714 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:32:10,715 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users 
WHERE users.name = %(name_1)s
2026-06-01 17:32:10,715 INFO sqlalchemy.engine.Engine [cached since 431.8s ago] {'name_1': 'Chris'}
2026-06-01 17:32:10,718 INFO sqlalchemy.engine.Engine SELECT posts.id AS posts_id, posts.user_id AS posts_user_id, posts.title AS posts_title 
FROM posts 
WHERE %(param_1)s = posts.user_id
2026-06-01 17:32:10,718 INFO sqlalchemy.engine.Engine [generated in 0.00051s] {'param_1': 1}
[Post(id=1, title='Python 뽀개기')]
2026-06-01 17:32:10,719 INFO sqlalchemy.engine.Engine ROLLBACK


#### User 와 Post 들 한번에 Insert

In [16]:
with SessionLocal.begin() as session:
    alice = User(name="Alice")
    alice.posts.append(Post(title="SQL 완성"))
    alice.posts.append(Post(title="데이터베이스 정복"))
    session.add(alice)
    # commit 시점에서 User INSERT, Post iNsert

    print_user(session)
    print_post(session)

    

[Users]
2026-06-01 17:36:30,934 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:36:30,935 INFO sqlalchemy.engine.Engine INSERT INTO users (name) VALUES (%(name)s)
2026-06-01 17:36:30,935 INFO sqlalchemy.engine.Engine [cached since 869.4s ago] {'name': 'Alice'}
2026-06-01 17:36:30,937 INFO sqlalchemy.engine.Engine INSERT INTO posts (user_id, title) VALUES (%(user_id)s, %(title)s)
2026-06-01 17:36:30,937 INFO sqlalchemy.engine.Engine [cached since 692s ago] {'user_id': 2, 'title': 'SQL 완성'}
2026-06-01 17:36:30,938 INFO sqlalchemy.engine.Engine INSERT INTO posts (user_id, title) VALUES (%(user_id)s, %(title)s)
2026-06-01 17:36:30,939 INFO sqlalchemy.engine.Engine [cached since 692s ago] {'user_id': 2, 'title': '데이터베이스 정복'}
2026-06-01 17:36:30,940 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users
2026-06-01 17:36:30,940 INFO sqlalchemy.engine.Engine [cached since 869.4s ago] {}
User(id=1, name='Chris')
User(id=2, name='Alice')

[Posts]
2026-06-01 17:36:30,9

In [18]:
with SessionLocal() as session:
    user = select_user_by_name(session, "Alice")
    print(f"{user.name} 의 작성글: ")
    # print(user.posts)   # SELECT 추가 실행

    for post in user.posts:
        print(f"  {post.title} (작성자: {post.author.name})")
    

2026-06-01 17:41:13,010 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:41:13,011 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users 
WHERE users.name = %(name_1)s
2026-06-01 17:41:13,012 INFO sqlalchemy.engine.Engine [cached since 974s ago] {'name_1': 'Alice'}
Alice 의 작성글: 
2026-06-01 17:41:13,013 INFO sqlalchemy.engine.Engine SELECT posts.id AS posts_id, posts.user_id AS posts_user_id, posts.title AS posts_title 
FROM posts 
WHERE %(param_1)s = posts.user_id
2026-06-01 17:41:13,014 INFO sqlalchemy.engine.Engine [cached since 542.3s ago] {'param_1': 2}
  SQL 완성 (작성자: Alice)
  데이터베이스 정복 (작성자: Alice)
2026-06-01 17:41:13,015 INFO sqlalchemy.engine.Engine ROLLBACK


여기서 `user.posts`에 접근할 때 자동으로 SELECT가 실행되는 것을 echo 로그로 확인할 수 있다. 이를 **lazy loading**이라 하며, 22장에서 자세히 다룬다.

In [19]:
# Back Populate 동작 확인

print(user)

User(id=2, name='Alice')


In [20]:
with SessionLocal.begin() as session:
    post = Post(title="안녕하세요 파이썬")
    user.posts.append(post)
    print('🟧', post.author)  # user (자동 설정됨)
    print('🟦', post.user_id) # 아직은 None
    session.add(post)

    

🟧 User(id=2, name='Alice')
🟦 None
2026-06-01 17:46:25,708 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:46:25,708 INFO sqlalchemy.engine.Engine INSERT INTO posts (user_id, title) VALUES (%(user_id)s, %(title)s)
2026-06-01 17:46:25,709 INFO sqlalchemy.engine.Engine [cached since 1287s ago] {'user_id': 2, 'title': '안녕하세요 파이썬'}
2026-06-01 17:46:25,710 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
# user # DetachedInstanceError 

In [22]:
with SessionLocal.begin() as session:
    user = select_user_by_name(session, 'Alice')
    print(user.posts)
    

2026-06-01 17:48:32,844 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-01 17:48:32,845 INFO sqlalchemy.engine.Engine SELECT users.id, users.name 
FROM users 
WHERE users.name = %(name_1)s
2026-06-01 17:48:32,845 INFO sqlalchemy.engine.Engine [cached since 1414s ago] {'name_1': 'Alice'}
2026-06-01 17:48:32,846 INFO sqlalchemy.engine.Engine SELECT posts.id AS posts_id, posts.user_id AS posts_user_id, posts.title AS posts_title 
FROM posts 
WHERE %(param_1)s = posts.user_id
2026-06-01 17:48:32,848 INFO sqlalchemy.engine.Engine [cached since 982.1s ago] {'param_1': 2}
[Post(id=2, title='SQL 완성'), Post(id=3, title='데이터베이스 정복'), Post(id=4, title='안녕하세요 파이썬')]
2026-06-01 17:48:32,850 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
"""
여기서 User 와 Post 테이블을 drop 해야 하나?

"""
None

---

## 17장. 일대다 / 다대일 관계

가장 흔한 관계 패턴이다. "한 사용자가 여러 글을 쓴다", "한 게시판에 여러 댓글이 달린다" 같은 구조다.

### 17.1 기본 형태

16장의 예제가 일대다/다대일의 전형이다. 다시 보자.

```python
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    posts: Mapped[List["Post"]] = relationship(back_populates="author")
    #     └──"one" 쪽: 컬렉션으로 표현


class Post(Base):
    __tablename__ = "posts"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    title: Mapped[str]
    
    author: Mapped["User"] = relationship(back_populates="posts")
    #      └──"many" 쪽: 단일 객체로 표현
```

User에서 보면 일대다(`one-to-many`), Post에서 보면 다대일(`many-to-one`)이다. 같은 관계의 두 면이다.

### 17.2 컬렉션 타입 선택

기본적으로 일대다 관계는 `List`로 표현된다. 다른 컬렉션 타입을 쓰고 싶다면 `Mapped`의 타입을 바꾼다.

```python
from typing import List, Set

# 리스트 (순서 있음, 중복 가능 - 기본)
posts: Mapped[List["Post"]] = relationship(back_populates="author")

# 셋 (순서 없음, 중복 불가)
tags: Mapped[Set["Tag"]] = relationship(back_populates="post")
```

대부분의 경우 `List`로 충분하다. 정렬이 필요하다면 `order_by`를 함께 지정한다.

```python
posts: Mapped[List["Post"]] = relationship(
    back_populates="author",
    order_by="Post.created_at.desc()",  # 문자열 또는 컬럼 표현식
)
```

### 17.3 캐스케이드(cascade)

#### 🟡cascade 가 없는 경우

In [ ]:
# cascade 가 없는 경우
with SessionLocal.begin() as session:
    🔹TODO
    

#### 🟡모든 테이블 삭제

In [ ]:
# 모든 테이블 삭제
Base.metadata.drop_all(engine)

부모 객체에 대한 작업을 자식에게 어떻게 전파할지 결정하는 옵션이다. 가장 자주 쓰이는 두 가지를 먼저 본다.

**기본 동작 (`save-update`)**

기본적으로 `save-update` 캐스케이드가 켜져 있다. 부모를 Session에 추가하면 컬렉션에 들어 있는 자식도 함께 Session에 추가된다.

```python
alice = User(name="Alice")
alice.posts.append(Post(title="A"))
alice.posts.append(Post(title="B"))

session.add(alice)  # Post 객체들도 자동으로 Session에 추가됨
session.commit()    # User와 Post 모두 INSERT
```

**`all, delete-orphan`**

가장 자주 쓰이는 캐스케이드 설정이다.

```python
posts: Mapped[List["Post"]] = relationship(
    back_populates="author",
    cascade="all, delete-orphan",
)
```

이 설정의 의미는 다음과 같다.

- `all`: 모든 표준 캐스케이드(`save-update`, `merge`, `refresh-expire`, `expunge`, `delete`) 적용
- `delete-orphan`: 부모와의 연결이 끊긴 자식은 자동 삭제됨

실제 동작 예:

```python
# 부모 삭제 → 자식도 함께 삭제됨 (delete cascade)
session.delete(alice)  # alice의 모든 posts도 DELETE됨

# 컬렉션에서 제거 → 자식 자동 삭제 (delete-orphan)
alice.posts.remove(some_post)  # some_post가 DELETE됨

# 부모 교체 → 이전 부모를 잃은 자식은 삭제됨
some_post.author = another_user
# alice는 some_post를 잃었고, some_post는 another_user의 자식이 됨
# 이전 author와의 연결이 끊긴 게 아니라 다른 부모로 옮긴 것이므로 삭제되지는 않음
```

**"고아"와 "분리"의 차이**

`delete-orphan`은 "부모가 **아예 없는** 자식"을 삭제한다. 다른 부모로 옮긴 경우는 고아가 아니므로 삭제되지 않는다. 부모가 한 명이어야 하는 강한 종속 관계(글-댓글, 주문-주문 항목 등)에 적합하다.

**캐스케이드 옵션 전체 정리**

| 옵션 | 의미 |
|---|---|
| `save-update` | 부모가 Session에 추가될 때 자식도 추가 (기본) |
| `delete` | 부모 삭제 시 자식도 삭제 |
| `delete-orphan` | 부모를 잃은 자식 자동 삭제 |
| `merge` | `Session.merge()` 시 자식에도 전파 (기본) |
| `expunge` | `Session.expunge()` 시 자식에도 전파 |
| `refresh-expire` | `Session.expire()` 시 자식에도 전파 |
| `all` | 위의 표준 캐스케이드 모두 (`delete-orphan` 제외) |

일반적인 권장:

- 강한 종속 관계(글-댓글, 주문-주문항목): `cascade="all, delete-orphan"`
- 약한 참조(사용자-팀): 캐스케이드 없음 (기본만)

### 17.4 데이터베이스 수준 ON DELETE

ORM의 `cascade="delete"`는 ORM이 직접 자식 DELETE를 실행한다. 반면 데이터베이스의 `ON DELETE CASCADE`는 DB가 알아서 처리한다. 두 가지는 다르다.

```python
class Post(Base):
    __tablename__ = "posts"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(
        ForeignKey("users.id", ondelete="CASCADE")  # DB 수준 cascade
    )
```

이 경우, 사용자를 직접 SQL DELETE해도 관련 글이 자동 삭제된다. ORM이 일일이 자식을 로드하지 않고도 정리되므로 대용량 데이터에서 효율적이다.

대규모 시스템에서는 다음과 같이 조합한다.

```python
posts: Mapped[List["Post"]] = relationship(
    back_populates="author",
    cascade="all, delete-orphan",
    passive_deletes=True,  # DB의 ON DELETE에 맡김
)
```

`passive_deletes=True`는 부모 삭제 시 ORM이 자식을 SELECT/DELETE하지 않고 DB의 ON DELETE에 위임한다. 단, DB에 실제로 `ondelete="CASCADE"`가 설정되어 있어야 한다.

### 🟡User, Post

In [ ]:
# 기존 모든 테이블 삭제
Base.metadata.drop_all(engine)
Base.metadata.clear()
Base.registry.dispose()

In [ ]:
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    posts: 🔹TODO
    

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r})"

class Post(Base):
    __tablename__ = "posts"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = 🔹TODO
    title: Mapped[str] = mapped_column(String(200))
    
    author: 🔹TODO
    
    def __repr__(self) -> str:
        return f"Post(id={self.id!r}, title={self.title!r})"    

In [ ]:
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(engine)

#### User, Post 데이터 추가

In [ ]:
with SessionLocal.begin() as session:
    alice = User(name="Alice")
    🔹TODO
    
    print_user(session)
    print_post(session)

#### User 삭제

In [ ]:
with SessionLocal.begin() as session:
    user = select_user_by_name(session, 'Alice')
    
    🔹TODO
    
    print_user(session)
    print_post(session)

### 17.5 일대일 관계

일대일은 일대다의 특수 케이스다. `relationship()`에 `uselist=False`를 주거나, `Mapped[T]`에서 컬렉션이 아닌 단일 객체를 표시한다.

```python
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    
    profile: Mapped["Profile | None"] = relationship(back_populates="user")


class Profile(Base):
    __tablename__ = "profiles"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"), unique=True)
    bio: Mapped[str]
    
    user: Mapped["User"] = relationship(back_populates="profile")
```

핵심은 두 가지다.

- `User` 쪽: `Mapped["Profile | None"]`로 단일 객체 표시 (또는 `uselist=False`)
- `Profile` 쪽: `user_id`에 `unique=True`로 한 사용자당 하나의 프로필만 허용

`unique=True`가 없으면 데이터베이스 수준에서 일대일이 강제되지 않는다.

---

## 18장. 다대다 관계

"한 사용자가 여러 그룹에 속하고, 한 그룹에 여러 사용자가 있는" 구조다. 두 테이블 사이에 직접 외래 키를 둘 수 없으므로 **연관 테이블(association table)** 이 필요하다.

### 18.1 단순 다대다: `secondary`

연관 테이블에 부가 정보가 없는 경우에는 `secondary` 옵션이 가장 간단하다.

```python
from sqlalchemy import Table, Column, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

class Base(DeclarativeBase):
    pass

# 연관 테이블 (Core Table로 정의)
user_group = Table(
    "user_group",
    Base.metadata,
    Column("user_id", ForeignKey("users.id"), primary_key=True),
    Column("group_id", ForeignKey("groups.id"), primary_key=True),
)

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    groups: Mapped[List["Group"]] = relationship(
        secondary=user_group, back_populates="users"
    )

class Group(Base):
    __tablename__ = "groups"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    users: Mapped[List["User"]] = relationship(
        secondary=user_group, back_populates="groups"
    )
```

연관 테이블은 ORM 클래스가 아니라 **Core의 `Table`**로 정의한다. 단순한 매핑 테이블이라면 별도의 클래스를 만들 필요가 없기 때문이다.

사용은 자연스럽다.

```python
with SessionLocal.begin() as session:
    admin_group = Group(name="admin")
    dev_group = Group(name="developer")
    
    alice = User(name="Alice")
    alice.groups.append(admin_group)
    alice.groups.append(dev_group)
    
    session.add(alice)
    # commit 시 users, groups, user_group 모두에 INSERT
```

조회도 자연스럽다.

```python
for user in alice.groups[0].users:
    print(user.name)
```

### 18.2 Association Object 패턴

연관 테이블에 추가 컬럼이 필요한 경우(예: 가입 시점, 권한 레벨)에는 연관 테이블 자체를 ORM 클래스로 정의한다. 이를 **Association Object 패턴**이라 부른다.

```python
from datetime import datetime
from sqlalchemy import ForeignKey, func

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    memberships: Mapped[List["Membership"]] = relationship(
        back_populates="user",
        cascade="all, delete-orphan",
    )


class Group(Base):
    __tablename__ = "groups"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    memberships: Mapped[List["Membership"]] = relationship(
        back_populates="group",
        cascade="all, delete-orphan",
    )


class Membership(Base):
    __tablename__ = "memberships"
    
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"), primary_key=True)
    group_id: Mapped[int] = mapped_column(ForeignKey("groups.id"), primary_key=True)
    role: Mapped[str] = mapped_column(default="member")
    joined_at: Mapped[datetime] = mapped_column(server_default=func.now())
    
    user: Mapped["User"] = relationship(back_populates="memberships")
    group: Mapped["Group"] = relationship(back_populates="memberships")
```

이제 `Membership`이라는 명시적 객체를 통해 관계를 다룬다.

```python
with SessionLocal.begin() as session:
    alice = User(name="Alice")
    admin_group = Group(name="admin")
    
    m = Membership(user=alice, group=admin_group, role="owner")
    session.add(m)
```

조회는 두 단계를 거친다.

```python
for membership in alice.memberships:
    print(f"{membership.group.name} - {membership.role}")
```

### 18.3 `secondary` vs Association Object 선택 기준

- **`secondary`**: 연관 테이블에 추가 컬럼이 없거나 절대 추가될 일이 없을 때
- **Association Object**: 추가 컬럼이 있거나, 향후 추가될 가능성이 있을 때

실무에서는 처음에 단순해 보여도 나중에 "언제 가입했는지" 같은 정보가 필요해지는 경우가 많다. 의심된다면 처음부터 Association Object로 가는 것이 안전하다.

### 18.4 두 패턴 함께 쓰기

`Membership`을 명시적으로 다루면서도, `user.groups`처럼 간편한 접근도 원할 수 있다. 이때는 두 관계를 모두 정의한다.

```python
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    
    memberships: Mapped[List["Membership"]] = relationship(
        back_populates="user", cascade="all, delete-orphan",
    )
    
    # 단축 접근용 (읽기 전용 권장)
    groups: Mapped[List["Group"]] = relationship(
        secondary="memberships",
        viewonly=True,
    )
```

`viewonly=True`가 중요하다. 같은 관계를 두 경로로 표현했으므로, 쓰기를 양쪽에서 시도하면 충돌이 난다. 한쪽은 읽기 전용으로 두는 것이 안전하다.

### 18.5 🟡종합 예제: 사용자-그룹 시스템

#### 🟡모든테이블 삭제

In [ ]:
# 모든 테이블 삭제
Base.metadata.drop_all(engine)
Base.metadata.clear()
Base.registry.dispose()

#### 🟡User, Group, Membership

In [ ]:
from datetime import datetime
from sqlalchemy import ForeignKey, func

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    memberships: 🔹TODO

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r})"    

class Group(Base):
    __tablename__ = "groups"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    
    memberships: 🔹TODO

    def __repr__(self) -> str:
        return f"Group(id={self.id!r}, name={self.name!r})"

class Membership(Base):
    __tablename__ = "memberships"
    
    user_id 🔹TODO
    group_id 🔹TODO
    role: Mapped[str] = mapped_column(String(50), default="member")
    joined_at: Mapped[datetime] = mapped_column(server_default=func.now())
    
    user 🔹TODO
    group 🔹TODO

    def __repr__(self) -> str:
        return f"Membership(user_id={self.user_id!r}, group_id={self.group_id!r}, role={self.role!r}, joined_at={self.joined_at!r})"    

In [ ]:
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(engine)    

#### User, Group 추가

In [ ]:
with SessionLocal.begin() as session:
    # 그룹 생성
    🔹TODO

    # # 사용자 + 멤버십 생성
    🔹TODO

    

    print_user(session)
    print_group(session)

# ※만약에 MySQL 에서 groups 테이블을 보려면 
# select * from `groups`;  라고 해야 한다.


#### Membership 추가

In [ ]:
with SessionLocal.begin() as session:
    🔹TODO

    # 방법1

    

In [ ]:
with SessionLocal.begin() as session:
    🔹TODO

    # 방법2


#### 특정 User가 속한 그룹과 역할

In [ ]:
# 조회: Alice가 속한 그룹과 역할
with SessionLocal() as session:
    🔹TODO

    

---

## 19장. 자기 참조 관계

같은 테이블 안에서 한 행이 다른 행을 참조하는 패턴이다. 카테고리의 부모-자식 관계, 직원-매니저 관계, 댓글의 답글 등이 해당한다.

### 19.1 인접 리스트(Adjacency List) 패턴

가장 단순한 자기 참조 구조다. 같은 테이블의 다른 행을 가리키는 외래 키 하나만 있으면 된다.

```python
from typing import List, Optional
from sqlalchemy import ForeignKey, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

class Base(DeclarativeBase):
    pass

class Category(Base):
    __tablename__ = "categories"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    parent_id: Mapped[int | None] = mapped_column(ForeignKey("categories.id"))
    
    # 자식들 (일대다)
    children: Mapped[List["Category"]] = relationship(
        back_populates="parent",
        cascade="all, delete-orphan",
    )
    
    # 부모 (다대일)
    parent: Mapped["Category | None"] = relationship(
        back_populates="children",
        remote_side=[id],  # 핵심!
    )
    
    def __repr__(self) -> str:
        return f"Category({self.name!r})"
```

핵심은 `remote_side=[id]`다. 같은 테이블 안에서 어느 쪽이 "remote(상대)"인지 SQLAlchemy가 알 수 없으므로 명시해야 한다.

- `parent_id`가 "local(이쪽)" 컬럼
- `id`가 "remote(저쪽)" 컬럼

부모 관계에서 `remote_side=[id]`는 "이 관계는 `id`가 외래 키 참조 대상이다 = 즉 부모를 가리킨다"는 의미다. 이를 통해 SQLAlchemy는 이 `parent` 관계가 다대일이라는 것을 안다.

### 19.2 사용 예제

```python
with SessionLocal.begin() as session:
    root = Category(name="Electronics")
    laptop = Category(name="Laptops", parent=root)
    phone = Category(name="Phones", parent=root)
    gaming_laptop = Category(name="Gaming Laptops", parent=laptop)
    
    session.add(root)
    # 모든 카테고리가 함께 INSERT됨

# 트리 순회
def print_tree(node: Category, depth: int = 0) -> None:
    print("  " * depth + node.name)
    for child in node.children:
        print_tree(child, depth + 1)

with SessionLocal() as session:
    root = session.execute(
        select(Category).where(Category.parent_id.is_(None))
    ).scalar_one()
    print_tree(root)
```

출력:
```
Electronics
  Laptops
    Gaming Laptops
  Phones
```

### 19.3 자기 참조 다대다

자기 자신과 다대다 관계도 가능하다. SNS의 팔로우-팔로워 관계가 전형적이다.

```python
from sqlalchemy import Table, Column, ForeignKey

follows = Table(
    "follows",
    Base.metadata,
    Column("follower_id", ForeignKey("users.id"), primary_key=True),
    Column("followee_id", ForeignKey("users.id"), primary_key=True),
)

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    # 내가 팔로우하는 사람들
    following: Mapped[List["User"]] = relationship(
        secondary=follows,
        primaryjoin=(id == follows.c.follower_id),
        secondaryjoin=(id == follows.c.followee_id),
        back_populates="followers",
    )
    
    # 나를 팔로우하는 사람들
    followers: Mapped[List["User"]] = relationship(
        secondary=follows,
        primaryjoin=(id == follows.c.followee_id),
        secondaryjoin=(id == follows.c.follower_id),
        back_populates="following",
    )
```

자기 참조 다대다는 SQLAlchemy가 조인 방향을 추론할 수 없으므로 `primaryjoin`과 `secondaryjoin`을 명시한다.

- `primaryjoin`: 부모(현재 객체)와 연관 테이블 사이의 조인 조건
- `secondaryjoin`: 연관 테이블과 자식(상대 객체) 사이의 조인 조건

```python
with SessionLocal.begin() as session:
    alice = User(name="Alice")
    bob = User(name="Bob")
    charlie = User(name="Charlie")
    
    alice.following.append(bob)
    alice.following.append(charlie)
    bob.following.append(alice)
    
    session.add_all([alice, bob, charlie])

with SessionLocal() as session:
    alice = session.execute(select(User).where(User.name == "Alice")).scalar_one()
    print("Alice가 팔로우하는 사람:", [u.name for u in alice.following])
    print("Alice의 팔로워:", [u.name for u in alice.followers])
```

---

## 20장. 상속 매핑

데이터베이스 테이블 구조 위에 클래스 상속 계층을 표현하는 방법이다. SQLAlchemy는 세 가지 전략을 제공한다.

### 20.1 세 가지 전략 비교

| 전략 | 테이블 구조 | 특징 |
|---|---|---|
| **Single Table** | 모든 클래스가 한 테이블 공유, 구분 컬럼으로 타입 식별 | 가장 단순, JOIN 없음. 서브클래스 전용 컬럼은 모두 NULL 허용 |
| **Joined Table** | 부모-자식 각자 테이블, 외래 키로 연결 | 정규화 잘됨. 조회 시 JOIN 필요 |
| **Concrete Table** | 각 클래스가 독립된 테이블, 공통 컬럼 중복 | 폴리모픽 조회가 까다로움 |

대부분의 경우 Single Table 또는 Joined Table을 사용한다. Concrete는 특수한 경우에만 쓰인다.

### 20.2 Single Table Inheritance

모든 클래스를 하나의 테이블에 매핑한다. 구분 컬럼(discriminator)으로 어떤 서브클래스인지 식별한다.

```python
from sqlalchemy import String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class Employee(Base):
    __tablename__ = "employees"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    type: Mapped[str] = mapped_column(String(20))  # 구분 컬럼
    
    __mapper_args__ = {
        "polymorphic_identity": "employee",
        "polymorphic_on": "type",
    }


class Engineer(Employee):
    # __tablename__ 없음! 같은 테이블 사용
    
    language: Mapped[str | None] = mapped_column(String(30))
    
    __mapper_args__ = {
        "polymorphic_identity": "engineer",
    }


class Manager(Employee):
    team_size: Mapped[int | None]
    
    __mapper_args__ = {
        "polymorphic_identity": "manager",
    }
```

핵심 포인트:

- 부모 클래스에 `polymorphic_on`으로 구분 컬럼 지정
- 각 클래스에 `polymorphic_identity`로 자신을 식별하는 값 지정
- 서브클래스에는 `__tablename__`을 적지 않음
- 서브클래스 전용 컬럼은 **반드시 nullable**이어야 함 (다른 서브클래스 행에서는 NULL이므로)

조회는 자동으로 폴리모픽하게 동작한다.

```python
with SessionLocal.begin() as session:
    session.add_all([
        Engineer(name="Alice", language="Python"),
        Engineer(name="Bob", language="Rust"),
        Manager(name="Charlie", team_size=5),
    ])

with SessionLocal() as session:
    # 모든 직원 조회 - 각자의 실제 클래스로 로드됨
    for emp in session.execute(select(Employee)).scalars():
        print(type(emp).__name__, emp.name)
    
    # 엔지니어만 조회
    engineers = session.execute(select(Engineer)).scalars().all()
    # 자동으로 WHERE type = 'engineer' 추가됨
```

**언제 쓰나**: 서브클래스 간 공통 컬럼이 많고, 차이 나는 컬럼이 적을 때. JOIN 없이 빠르게 조회되어야 할 때.

**단점**: 서브클래스 전용 컬럼이 많아지면 테이블이 sparse(NULL 가득)해진다.

### 20.3 Joined Table Inheritance

각 클래스가 독립된 테이블을 갖고, 외래 키로 연결된다.

```python
class Employee(Base):
    __tablename__ = "employees"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    type: Mapped[str] = mapped_column(String(20))
    
    __mapper_args__ = {
        "polymorphic_identity": "employee",
        "polymorphic_on": "type",
    }


class Engineer(Employee):
    __tablename__ = "engineers"
    
    id: Mapped[int] = mapped_column(ForeignKey("employees.id"), primary_key=True)
    language: Mapped[str] = mapped_column(String(30))
    
    __mapper_args__ = {
        "polymorphic_identity": "engineer",
    }


class Manager(Employee):
    __tablename__ = "managers"
    
    id: Mapped[int] = mapped_column(ForeignKey("employees.id"), primary_key=True)
    team_size: Mapped[int]
    
    __mapper_args__ = {
        "polymorphic_identity": "manager",
    }
```

핵심 포인트:

- 각 서브클래스에 `__tablename__`을 정의
- 서브클래스의 `id`는 부모 테이블의 `id`를 참조하는 외래 키이면서 동시에 자신의 기본 키
- 서브클래스 전용 컬럼은 NOT NULL 가능 (자신의 테이블에 있으므로)

조회 시 SQLAlchemy는 자동으로 JOIN을 생성한다.

```python
session.execute(select(Engineer)).scalars().all()
# SELECT employees.*, engineers.*
# FROM employees JOIN engineers ON employees.id = engineers.id
# WHERE employees.type = 'engineer'
```

**언제 쓰나**: 서브클래스 간 차이가 크고, 정규화가 중요할 때. 각 서브클래스가 자기 컬럼에 NOT NULL 제약을 갖고 싶을 때.

**단점**: 모든 조회에 JOIN이 필요해 약간의 성능 비용. 폴리모픽 로딩이 N+1 문제를 일으킬 수 있음(`with_polymorphic`로 해결).

### 20.4 폴리모픽 로딩과 `with_polymorphic`

Joined 상속에서 base 클래스로 조회하면, 기본적으로 base 테이블만 SELECT한다. 서브클래스 전용 컬럼은 그 객체에 접근할 때 별도 쿼리로 로드된다(lazy). 이로 인해 N+1이 발생할 수 있다.

이를 한 번에 해결하려면 `with_polymorphic`을 쓴다.

```python
from sqlalchemy.orm import with_polymorphic

with SessionLocal() as session:
    # 모든 서브클래스를 한 번에 JOIN
    employees = session.execute(
        select(with_polymorphic(Employee, "*"))
    ).scalars().all()
    
    for emp in employees:
        if isinstance(emp, Engineer):
            print(f"{emp.name}: {emp.language}")
        elif isinstance(emp, Manager):
            print(f"{emp.name}: team of {emp.team_size}")
```

또는 매퍼 수준에서 기본 폴리모픽 로딩을 설정할 수도 있다.

```python
class Employee(Base):
    ...
    __mapper_args__ = {
        "polymorphic_identity": "employee",
        "polymorphic_on": "type",
        "polymorphic_load": "selectin",  # 서브클래스 자동 selectin 로딩
    }
```

### 20.5 어떤 전략을 선택할까

- 가벼운 차이, 빠른 조회 우선: **Single Table**
- 명확한 분리, 정규화 우선: **Joined Table**
- 클래스가 완전히 독립적이고 폴리모픽 조회가 거의 없음: **Concrete Table** (드뭄)

대부분의 실무 시나리오는 Joined Table이 더 깔끔하다. 다만 성능에 민감한 핫 패스에서 Single Table을 선택하는 경우도 많다. 정답은 도메인에 따라 다르다.

---

## Part 5 마무리

여기까지 따라왔다면 다음을 할 수 있게 되었다.

- `relationship()`과 `back_populates`로 양방향 관계 정의
- 일대다, 다대일, 일대일 관계 매핑과 캐스케이드 설정
- 다대다 관계: `secondary`를 이용한 단순 매핑과 Association Object 패턴
- 자기 참조 관계: 인접 리스트, 자기 참조 다대다 (`primaryjoin`/`secondaryjoin`)
- 상속 매핑의 세 가지 전략과 각각의 트레이드오프

다음 Part 6에서는 이 관계들을 **효율적으로 쿼리하는 법**을 다룬다. 17장에서 잠시 언급한 lazy loading의 N+1 문제, `selectinload`/`joinedload` 같은 로딩 전략, 그리고 복잡한 JOIN, 서브쿼리, CTE를 ORM 스타일로 작성하는 방법을 본다.
